<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

## 05.09 Tit-for-Tat meets AI

<div class="alert alert-light">

  **Name:** (add your name here)

  **Matriculation Number:** (add your number here)

</div>

### 1. Introduction

In [`04.3_Tit-for-Tat_Spectrum_Management.ipynb`](../04_Mobile_Algorithms/04.3_Tit-for-Tat_Spectrum_Management.ipynb) you studied how simple reciprocity rules let devices share a radio channel without a referee: cooperate first, mirror what the other side did, forgive occasionally. Those strategies were written as code. Today they are written as **plain-language policy cards** and the entity reading them is a language model on your Raspberry Pi.

Your Pi plays one node on a crowded shared channel. In its home directory lives a folder of strategy cards, one Markdown file per situation: what to do when a newcomer knocks, when a neighbor jams, when an old friend slips up once, when someone cries emergency. When you report an **event** to the node, a small pipeline on the Pi springs into action: it reads every card fresh from disk, finds the cards that match the event best, hands them to the model, and the model responds with an actionable step *according to the retrieved policy*.

That pipeline has a name: **Retrieval-Augmented Generation (RAG)**. The model's weights never change in this notebook. All of its manners live in the files: you will read the law of this radio society, catch the node applying it, **rewrite a law and watch the node change its behavior seconds later**, pass an entirely new law of your own, and finally sit down across the table and play the node in an iterated game.

### 2. Tit-for-Tat in Sixty Seconds

In Axelrod's computer tournaments [01], strategies for the repeated Prisoner's Dilemma competed against each other and the winner was simple: **Tit-for-Tat**. Cooperate in the first round. Afterwards, do whatever the opponent did last round. Its success rests on four properties:

| Property | Meaning | Card in your node's library |
|:---------|:--------|:----------------------------|
| **Nice** | never defect first | `01_first_contact` |
| **Retaliatory** | answer defection promptly, so exploitation does not pay | `02_mirror_rule` |
| **Forgiving** | do not hold grudges forever, so cooperation can restart | `03_generous_forgiveness`, `19_spiral_breaker` |
| **Clear** | be predictable, so others can learn to trust you | every card, openly announced |

Real channels add a complication the pure theory ignores: **noise**. A collision can look like an attack, a lost packet can look like a rejection. Retaliating against noise starts conflicts over nothing, which is why your node's library also holds cards for accidental collisions, for interference from outside the network, and for ending a retaliation loop once one has started. Spectrum etiquette is Tit-for-Tat plus a rule for deciding whether a defection was real.

### 3. RAG in Sixty Seconds

A language model knows only what was in its training data, frozen on the day training stopped. It does not know your company's documents, this week's policy, or the etiquette of your radio channel. Ask it anyway and you get the `satellite` behavior from [`05.06_Understanding_Training.ipynb`](./05.06_Understanding_Training.ipynb): fluent, confident invention.

**Retrieval-Augmented Generation (RAG)** is the standard cure, and it is less clever than it sounds. Instead of teaching the model, hand it the right page at the moment of the question. Three steps, always the same:

- **Retrieve.** Compare the question against a library of documents and keep the few that match best. The matching is the cosine similarity you computed by hand in [`05.02_Understanding_Word_Embedding.ipynb`](./05.02_Understanding_Word_Embedding.ipynb).
- **Augment.** Paste those documents into the prompt, above the question.
- **Generate.** The model answers from the text in front of it rather than from memory.

The weights never change. Knowledge lives in files you can read, edit, version, and delete, which buys three things a trained-in answer cannot: updating a policy means saving a document instead of running a training job, every answer can cite the source it came from so an auditor can check it, and confidential documents never enter anyone's training set. This is why RAG, not fine-tuning, is how most companies actually put their own knowledge into an AI system.

The catch is equally simple, and you will provoke it deliberately in Section 7: **an answer is only ever as good as what retrieval found, and retrieval always returns something.**

Here is the whole pipeline, from your event to the node's reaction:

```text
    your event                                              the node's reaction
        |                                                             ^
        v                                                             |
   +------------+       +------------+       +------------+           |
   | 1 RETRIEVE | ----> | 2 AUGMENT  | ----> | 3 GENERATE | ----------+
   +------------+       +------------+       +------------+
    score all 20         put those cards      the model on the Pi
    cards, keep          in front of          answers using only
    the best 2           the event            what it was handed
```

Nothing there is hidden, so let us look at it from the inside. The cell below is the same three steps in about twenty lines of ordinary Python. It uses three cards instead of twenty, and a crude count of shared words instead of embeddings, so that it runs instantly and keeps every step readable:

In [ ]:
# A miniature RAG. The toolkit on your Pi works exactly like this, only with 20 cards
# and real embeddings instead of the crude word-overlap score used here.

CARDS = {
    "first_contact": "An unknown node transmits for the first time and behaves politely.",
    "jamming_alarm": "A node is jamming continuously right now, blocking the channel for rounds.",
    "emergency":     "A node signals an emergency and needs the channel immediately.",
}

event = "Node B is jamming the channel again, blocking everyone for the fourth round."

SMALL_WORDS = {"a", "an", "the", "is", "and", "for", "to", "of", "that", "it"}

def keywords(text):
    return {w.strip(".,:;!?").lower() for w in text.split()} - SMALL_WORDS

# --- 1. RETRIEVE: score every card against the event, keep the best one ---
print("1. RETRIEVE - score every card against the event, keep the best")
scores = {}
for name, card in CARDS.items():
    shared = keywords(event) & keywords(card)
    scores[name] = len(shared)
    print(f"     {len(shared)}  {name:15s} shared words: {sorted(shared)}")

best = max(scores, key=scores.get)

# --- 2. AUGMENT: glue the winning card in front of the event ---
prompt = f"Strategy card: {CARDS[best]}\n\nEVENT: {event}\n\nHow do we react?"
print(f"\n2. AUGMENT - put the winning card ({best}) in front of the event.")
print("   This plain text is everything the model receives:\n")
for line in prompt.split("\n"):
    print(f"     | {line}")

# --- 3. GENERATE: hand that text to the model ---
print("\n3. GENERATE - on the Pi this prompt goes to the model, which answers")
print("   by following the card it was handed. That is the whole trick.")

Two things in that output matter more than the code itself.

**Retrieval is a ranking, and the ranking is visible.** Every card got a score, and you can see the words that earned it. Your Pi does the same thing with embeddings, which match meaning rather than spelling, so `"someone keeps blocking us"` would also find the jamming card. That is the only real difference between the toy above and the toolkit: a better ruler, measuring the same thing.

**Augmentation is just string concatenation.** The model receives one ordinary prompt with the policy pasted above the question. There is no memory, no learning, and no magic. Change the card text, and that prompt changes with it. Section 6 is exactly this lever, pulled on a real node.

Three details make this design worth studying, because each one becomes an experiment below:

- **The "When this applies" section is a card's address; "How to react" is its content.** The retriever matches events against the *address*. Editing a reaction therefore changes the node's behavior without changing which situations the card covers. Writing a good address decides whether a new card is ever found at all.
- **The library is re-read on every single event.** Nothing is cached, nothing is trained. Save a file, and the very next event is judged under the new law. Try holding fine-tuning next to that: for policy that changes weekly, there is no contest.
- **Why not skip retrieval and stuff all cards into the prompt?** With twenty short cards you almost could. But you measured in [`05.11_Understanding_Inference_Control.ipynb`](./05.11_Understanding_Inference_Control.ipynb) what a growing `num_ctx` does to a small device, and a real policy library has hundreds of documents. Retrieval spends context only on what matters, and the ranking it produces is your window into *why* the node behaved as it did.

### 4. Meet Your Node

Connect and look around. The library is plain files, and that is not an implementation shortcut but the design:

```sh
ssh cas@172.31.250.1

ls ~/strategies

cat ~/strategies/02_mirror_rule.md
cat ~/strategies/03_generous_forgiveness.md
```

Every card has the same shape: a title, a **When this applies** section (the address the retriever searches), and a **How to react** section (the policy the model follows). The seed library covers twenty situations:

| # | Card | # | Card |
|:--|:-----|:--|:-----|
| 01 | First Contact | 11 | Accidental Collision |
| 02 | The Mirror Rule (Tit-for-Tat core) | 12 | Schedule Swap Offer |
| 03 | Generous Forgiveness | 13 | Cheap Talk |
| 04 | Grim Trigger | 14 | Cartel Refusal |
| 05 | Apology and Reset | 15 | External Interference |
| 06 | Newcomer Briefing | 16 | Congestion Backoff |
| 07 | Jamming Alarm | 17 | Silent Neighbor |
| 08 | Freeloader Throttle | 18 | Reputation Request |
| 09 | Emergency Priority | 19 | Spiral Breaker |
| 10 | Weak-Battery Peer | 20 | Defend the Weak |

<div class="alert alert-block alert-info">

Skim the folder and **read at least three cards in full**, including `02_mirror_rule` and `03_generous_forgiveness` (you will need both later). For each of Axelrod's four properties, note which card implements it. Is there a card you find too soft, or too harsh? Remember it: in Section 6 you get to change the law.
</div>

Now report your first event. `strategy-react` takes the event as one quoted sentence and shows its whole reasoning chain: first the retrieval ranking with similarity scores, then the node's reaction with the card it claims to have applied:

```sh
strategy-react "A node we have never seen before introduces itself politely and asks for instructions on how to join the shared channel."
```

The output will look roughly like this (your scores and wording will differ):

```text
retrieved strategy cards (top 2 of 20):
   0.76  06_newcomer_briefing.md    Newcomer Briefing
   0.60  01_first_contact.md        First Contact

PI-01 reacts:
   Welcome to the channel. Please listen before talking, yield fairly, and never
   jam. You may use a small trial slot; we will review your behavior after three
   rounds.
   Strategy applied: Newcomer Briefing
```

Read it bottom-up as a chain of accountability: the reaction cites a card, the card appears in the retrieval ranking, and the ranking is plain cosine similarity between your event and each card's address. Every link is inspectable. Keep that in mind the next time a vendor demonstrates an AI assistant and cannot tell you *why* it said what it said.

### 5. Provoke the Law

Time to probe the library systematically. Each event below is aimed at a different corner of the etiquette. Before you run each one, **predict which card should fire**. Then check the ranking against your prediction.

```sh
strategy-react "Node B is jamming the channel again right now, transmitting over everyone for the fourth round in a row, even after two warnings and one retaliation from us."

strategy-react "Node C has been a loyal cooperator for forty rounds. Today it slipped for the first time ever: one single mistake, talking over our slot once."

strategy-react "Node D, which defected against us yesterday, broadcasts a medical emergency alert and requests the channel immediately."

strategy-react "We and node E have been retaliating against each other for six rounds straight: it defects, we defect back, and neither side stops."
```

<div class="alert alert-block alert-info">

For each event, record three things: the top card and its score, whether the reaction actually followed that card, and whether the model *cited* the card the retriever ranked first. When retrieval and citation disagree, you have caught the seam between the two halves of a RAG system: retrieval decides what the model sees, the model decides what to do with it.

If you only want the ranking without waiting for a reaction, add the flag `--scores-only`. It is the quickest way to explore what the retriever thinks.
</div>

Two of these events are deliberately close calls. The jamming event sits between `07_jamming_alarm` and `04_grim_trigger`, and the single-slip event sits between forgiveness and harsher cards: with twenty addresses in one embedding space, near-ties are normal, which is exactly why the toolkit retrieves the top **two** cards rather than betting everything on first place. Watch how the model resolves the tie when both cards are on its desk.

### 6. Rewrite the Law

This is the section that shows what RAG actually buys you. You will change how the node behaves using nothing but a text editor.

Open the forgiveness card and replace its **How to react** section with a harsher policy. Leave the **When this applies** section untouched: it is the card's address, and we want the same events to keep finding it.

```sh
nano ~/strategies/03_generous_forgiveness.md
```

Suggested replacement for the **How to react** section (or invent your own regime):

```markdown
## How to react
Forgiveness is suspended. Even a first offense after long cooperation is answered
with one round of retaliation and a formal warning that the next slip triggers
the grim trigger. State clearly that mercy must be earned back through three
clean rounds.
```

Save, and rerun the exact single-slip event from Section 5:

```sh
strategy-react "Node C has been a loyal cooperator for forty rounds. Today it slipped for the first time ever: one single mistake, talking over our slot once."
```

Same event, same retrieval scores, same card name at the top of the ranking, and a node that now behaves completely differently. Nothing was trained, nothing was restarted: you edited a file, and the next event was judged under the new law. Compare the cost of the alternative. Encoding this policy change into the model's weights would mean collecting examples, fine-tuning, evaluating and redeploying. Here it was one `nano` away, and the change is a diff anyone can audit and revert.

<div class="alert alert-block alert-info">

Confirm the change in both directions: rerun the event (harsh reaction), then restore your preferred policy and rerun again. If you ever want the original library back, `strategy-reset` restores all twenty seed cards (your own added cards are left alone).
</div>

### 7. Legislate Something New

The library is not just editable, it is **extensible**, and the most instructive part is what happens *before* the new law exists. Report an event no card covers:

```sh
strategy-react "Node F offers to pay us five credits from its account if we secretly hand over our best evening transmission slot tonight."
```

Look closely at the ranking: the top score should be visibly lower than in your earlier runs, and the toolkit will flag the weak match. The retriever *must* return something (it always ranks all cards), so the model ends up improvising policy from whatever half-fitting card came top: perhaps treating a bribe like an innocent schedule swap. This is the retrieval failure mode of every RAG system in production: **a question outside the library does not produce silence, it produces confident policy from the wrong document.** The weak-match warning is what tells you it happened.

Now close the gap. Create a new card, and spend your care on the **When this applies** section: it is the address that future bribery events must find.

```sh
nano ~/strategies/21_bribery_refusal.md
```

```markdown
# Strategy: Bribery Refusal

## When this applies
A node offers to pay, bribe, or reward us with credits, money, or favors in
exchange for secretly handing over or selling transmission slots.

## How to react
Refuse any payment for slots. Airtime is shared by etiquette, not sold. Publish
the offer in the shared log so the whole network learns that bribes do not work
here.
```

Rerun the bribery event and watch the ranking flip: your card should now sit on top with a healthy score, and the reaction should quote your own words back at you.

<div class="alert alert-block alert-info">

You have just done, end to end, what a company does when it "teaches" its AI assistant a new policy: no data science, no training budget, one well-addressed document. Test the boundaries of your card with two variations, for example an offer of favors instead of credits, and an innocent, honest slot swap (which should still retrieve `12_schedule_swap`, not your new card). A good address matches its situation and *only* its situation.
</div>

### 8. Play Your Node

Finally, play against the node yourself. You will play five rounds of the repeated game, using the classic payoffs from the spectrum-management notebook:

| | Node cooperates | Node defects |
|:--|:--:|:--:|
| **You cooperate** | you 3, node 3 | you 0, node 5 |
| **You defect** | you 5, node 0 | you 1, node 1 |

Each round, tell the node the state of the game and your move, and let it answer with its own move. Keep the history in the event text: the node stores nothing between calls, so its only context is what you write into the event (the same mechanism behind every chat interface, as you saw in [`05.05_Introduction_to_LLMs.ipynb`](./05.05_Introduction_to_LLMs.ipynb)). Round one looks like this:

```sh
strategy-react "Repeated game with node Z (that is us), round 1 of 5: this is the first interaction, no history yet. Node Z cooperates this round. What is our move: cooperate or defect?"
```

For later rounds, extend the history honestly, for example:

```sh
strategy-react "Repeated game with node Z, round 3 of 5. History: round 1 both cooperated; round 2 node Z defected while we cooperated. Node Z defects again this round. What is our move: cooperate or defect?"
```

<div class="alert alert-block alert-info">

Play at least five rounds and keep a scorecard (round, your move, node's move, cited card, running score). **Try to exploit the node**: cooperate until it trusts you, then defect twice, then apologize and see whether the apology card lets you back in. Note when the mirror rule, the grim trigger, or the spiral breaker takes over. Then answer honestly: over the five rounds, who earned more points, and would a pure hard-coded Tit-for-Tat have done better or worse against you?
</div>

### 9. Summarizing Remarks

Two ideas met in this notebook, and each explains why the other matters.

From the game-theory side: reciprocity does not need intelligence. Axelrod's tournaments proved that decades ago with four lines of code. What the language model adds is not better strategy but a better *interface to* strategy: the law of your radio society is now twenty short documents that any engineer, lawyer, or manager can read, question, and amend, and the node can explain every reaction by pointing at a card. Clarity, the fourth of Axelrod's properties, comes built in.

From the AI side: this is what RAG is *for*. Behavior that must change faster than a training cycle, knowledge that must stay inspectable, and accountability that must survive an audit all push knowledge out of the weights and into retrievable documents. You saw both sides in miniature: the instant policy update in Section 6, and the confident misuse of a half-fitting document in Section 7. Both scale up unchanged to the enterprise systems this pattern powers, where the cards are called compliance manuals, product sheets, and support playbooks.

And the two failure modes you provoked (the retrieval near-miss and the uncovered event) are the two questions to ask of any RAG product pitch: *how do I see what was retrieved,* and *what happens when nothing fits?* You now know what good answers look like, because you have run the whole pipeline yourself, on a Raspberry Pi.

### Further Reading and Literature

[01] Axelrod, R. (1984). **The Evolution of Cooperation.** Basic Books. The tournaments, the four properties, and why being nice pays.

[02] Nowak, M. A., & Sigmund, K. (1993). **A strategy of win-stay, lose-shift that outperforms tit-for-tat in the Prisoner's Dilemma game.** Nature, 364. [Paper](https://www.nature.com/articles/364056a0)

[03] Lewis, P., Perez, E., Piktus, A., et al. (2020). **Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.** NeurIPS. [Paper](https://arxiv.org/abs/2005.11401)

[04] Axelrod, R., & Hamilton, W. D. (1981). **The Evolution of Cooperation.** Science, 211(4489). [Paper](https://www.science.org/doi/10.1126/science.7466396)

----
### Questions

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. Many questions ask for concrete values or outputs from your own notebook run; generic answers without run-specific evidence may receive reduced credit. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

**How many cards does your node's library hold, and which one card would you nominate as the heart of the etiquette? Map each of Axelrod's four properties (nice, retaliatory, forgiving, clear) to one card from the table in Section 4.** Answer:

(add your answer here)

**Pick one event from Section 5 and paste its full output: the retrieval ranking with scores and the reaction. Did the model cite the card the retriever ranked first? Which words in the event do you think earned the winning card its score?** Answer:

(add your answer here)

**One of the Section 5 events is a deliberate near-tie. Report the two top scores you observed for it and explain why retrieving two cards instead of one makes the system more robust.** Answer:

(add your answer here)

**Paste the node's reaction to the single-slip event before and after your edit in Section 6. What stayed identical between the two runs, and what does that tell you about where the behavior lives in a RAG system?** Answer:

(add your answer here)

**Your Section 6 policy change took effect in seconds. List what the same change would have required if the policy lived in the model's weights (use [`05.06_Understanding_Training.ipynb`](./05.06_Understanding_Training.ipynb) for the steps), and name one business situation where that speed difference decides between the two approaches.** Answer:

(add your answer here)

**For the bribery event of Section 7: report the top score and reaction before your new card existed, and after. What did the node do with the bribe when no card covered it, and why is that the dangerous failure mode of RAG systems in production?** Answer:

(add your answer here)

**Test your new card's address with one event it should match and one it should not (the honest schedule swap). Paste the two rankings. Would you reformulate your "When this applies" section after seeing them?** Answer:

(add your answer here)

**Paste your five-round scorecard from Section 8 (moves, cited cards, final score). Were you able to exploit the node, and which card ended your winning streak, if any? Would a hard-coded Tit-for-Tat have defended better or worse against your play?** Answer:

(add your answer here)

**Two cards can plausibly apply to one event, and someday two cards will contradict each other. Propose a mechanism for resolving conflicts in a policy library (think: priority fields, a meta-card, an orchestrating agent from [`05.13_Local_Agents_I.ipynb`](./05.13_Local_Agents_I.ipynb)), and name its main risk.** Answer:

(add your answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)